In [49]:
import math
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.model_selection import train_test_split


In [50]:
np.random.seed(42)
tf.random.set_seed(42)

In [51]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 25
EPOCHS_FINETUNE = 15

In [52]:
base_dir = './dataset/'
class_names = sorted(
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d))
)
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}

image_paths = []
labels = []

In [ ]:
for class_name in class_names:
    class_dir = os.path.join(base_dir, class_name)
    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name)
        image_paths.append(image_path)
        labels.append(class_name)

df = pd.DataFrame({'image_path': image_paths, 'label': labels})

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [54]:
def build_dataset(df, shuffle=False, augment=False):
    paths = df['image_path'].astype(str).values
    y = np.array([label_to_index[l] for l in df['label']], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42, reshuffle_each_iteration=True)

    def decode(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        def aug(image, label):
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=0.15)
            image = tf.image.random_contrast(image, 0.85, 1.15)
            image = tf.image.random_saturation(image, 0.85, 1.15)
            image = tf.image.random_hue(image, 0.03)
            return tf.clip_by_value(image, 0.0, 1.0), label

        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)

    return ds


train_ds = build_dataset(train_df, shuffle=True, augment=True).batch(BATCH_SIZE).repeat().prefetch(
    tf.data.AUTOTUNE
)
val_ds = build_dataset(val_df, shuffle=False, augment=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = build_dataset(test_df, shuffle=False, augment=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [55]:
inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = layers.Lambda(lambda t: preprocess_input(t * 255.0))(inputs)
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_tensor=x,
    include_preprocessing=False,
    pooling=None,
)
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [56]:
steps_per_epoch = max(1, math.ceil(len(train_df) / BATCH_SIZE))
val_steps = max(1, math.ceil(len(val_df) / BATCH_SIZE))

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_HEAD,
    validation_data=val_ds,
    validation_steps=val_steps,
    callbacks=callbacks_phase1,
)

base_model.trainable = True
fine_tune_at = max(1, len(base_model.layers) - 55)
for i, layer in enumerate(base_model.layers):
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = i >= fine_tune_at

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-8,
        verbose=1,
    ),
]

history_finetune = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    initial_epoch=len(history.epoch),
    epochs=len(history.epoch) + EPOCHS_FINETUNE,
    validation_data=val_ds,
    validation_steps=val_steps,
    callbacks=callbacks_phase2,
)

Epoch 1/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 219ms/step - accuracy: 0.1036 - loss: 2.6606 - val_accuracy: 0.1285 - val_loss: 2.6207
Epoch 2/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 217ms/step - accuracy: 0.1345 - loss: 2.5716 - val_accuracy: 0.2361 - val_loss: 2.4537
Epoch 3/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 222ms/step - accuracy: 0.2054 - loss: 2.4320 - val_accuracy: 0.2674 - val_loss: 2.2669
Epoch 4/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 241ms/step - accuracy: 0.2450 - loss: 2.3542 - val_accuracy: 0.2743 - val_loss: 2.2404
Epoch 5/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 244ms/step - accuracy: 0.2880 - loss: 2.2332 - val_accuracy: 0.3299 - val_loss: 2.1962
Epoch 6/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 233ms/step - accuracy: 0.3003 - loss: 2.1672 - val_accuracy: 0.3681 - val_loss: 2.0173
Epoch 7/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 233ms/step - accuracy: 0.3210 - loss: 2.0852 - val_accuracy: 0.3924 - val_loss: 2.0173
Epoch 8/30
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 234ms/step - accuracy: 0.3729 - loss: 1.9604 - val_accuracy: 0.

In [57]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.8638 - loss: 0.7786
Test accuracy: 0.8638


In [58]:
model.save('animal_classification_model.keras')